# Imports

In [ ]:
!pip install kagglehub
!pip install rouge-score

In [ ]:
import kagglehub
import os
import json
import re
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from rouge_score import rouge_scorer
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
path = kagglehub.dataset_download(
    "gowrishankarp/newspaper-text-summarization-cnn-dailymail"
)

print("Dataset path:", path)
files = os.listdir(path)
print("Number of files:", len(files))

Using Colab cache for faster access to the 'newspaper-text-summarization-cnn-dailymail' dataset.
Dataset path: /kaggle/input/newspaper-text-summarization-cnn-dailymail
Number of files: 1


#Dataset Splitting and Cleaning

In [ ]:
data_dir = os.path.join(path, "cnn_dailymail")

train_file = os.path.join(data_dir, "train.csv")
val_file   = os.path.join(data_dir, "validation.csv")
test_file  = os.path.join(data_dir, "test.csv")

print(os.listdir(data_dir))

['validation.csv', 'train.csv', 'test.csv']


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
import csv # Import the csv module to handle CSV files

def load_data(csv_file_path, limit):
    articles, summaries = [], []
    with open(csv_file_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f) # Read CSV data with DictReader
        for i, row in enumerate(reader):
            if i >= limit:
                break
            # Assuming the CSV has 'article' and 'highlights' columns
            articles.append(clean_text(row["article"]))
            summaries.append(clean_text(row["highlights"]))
    return articles, summaries

# Correct the path for val_file for case sensitivity
# 'data_dir' is assumed to be in scope from a previous cell.
val_file = os.path.join(data_dir, "validation.csv") # Fix capitalization from 'Validation.csv'

train_articles, train_summaries = load_data(train_file, limit=8000)
val_articles, val_summaries     = load_data(val_file, limit=1000)
test_articles, test_summaries   = load_data(test_file, limit=1000)

print("Train:", len(train_articles))
print("Val:", len(val_articles))
print("Test:", len(test_articles)) # Corrected typo: should be test_articles

Train: 8000
Val: 1000
Test: 1000


#Tokenization and Seq2Seq model definition

In [ ]:
def tokenize(text):
    return text.split()

counter = Counter()
for text in train_articles + train_summaries:
    counter.update(tokenize(text))

VOCAB_SIZE = 10000

word2idx = {
    "<pad>": 0,
    "<sos>": 1,
    "<eos>": 2,
    "<unk>": 3
}

for i, (word, _) in enumerate(counter.most_common(VOCAB_SIZE - 4), start=4):
    word2idx[word] = i

idx2word = {i: w for w, i in word2idx.items()}

In [ ]:
MAX_ART_LEN = 500
MAX_SUM_LEN = 70

def encode(text, max_len):
    tokens = tokenize(text)
    ids = [word2idx.get(w, 3) for w in tokens][:max_len]
    return ids + [0] * (max_len - len(ids))

X_train = torch.tensor([encode(a, MAX_ART_LEN) for a in train_articles])
y_train = torch.tensor([[1] + encode(s, MAX_SUM_LEN - 2) + [2] for s in train_summaries])

X_val = torch.tensor([encode(a, MAX_ART_LEN) for a in val_articles])
y_val = torch.tensor([[1] + encode(s, MAX_SUM_LEN - 2) + [2] for s in val_summaries])

In [ ]:
train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=16
)


In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell


In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        seq_len = encoder_outputs.size(1)
        hidden = hidden[-1].unsqueeze(1).repeat(1, seq_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        return torch.softmax(self.v(energy).squeeze(2), dim=1)


In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, attention):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attention = attention
        self.lstm = nn.LSTM(hidden_dim + embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)

    def forward(self, x, hidden, cell, encoder_outputs):
        x = x.unsqueeze(1)
        embedded = self.embedding(x)

        attn_weights = self.attention(hidden, encoder_outputs)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)

        lstm_input = torch.cat((embedded, context), dim=2)
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))

        output = output.squeeze(1)
        context = context.squeeze(1)

        return self.fc(torch.cat((output, context), dim=1)), hidden, cell


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size, trg_len = trg.shape
        vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(batch_size, trg_len, vocab_size).to(src.device)

        encoder_outputs, hidden, cell = self.encoder(src)
        input_token = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(
                input_token, hidden, cell, encoder_outputs
            )
            outputs[:, t] = output
            input_token = trg[:, t] if torch.rand(1).item() < teacher_forcing_ratio else output.argmax(1)

        return outputs


#Training

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

encoder = Encoder(VOCAB_SIZE, 256, 512)
attention = Attention(512)
decoder = Decoder(VOCAB_SIZE, 256, 512, attention)

model = Seq2Seq(encoder, decoder).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)

In [ ]:
def train_epoch(loader):
    model.train()
    total_loss = 0

    for src, trg in loader:
        src, trg = src.to(DEVICE), trg.to(DEVICE)

        optimizer.zero_grad()
        output = model(src, trg)

        output = output[:, 1:].reshape(-1, VOCAB_SIZE)
        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def evaluate_epoch(loader):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for src, trg in loader:
            src, trg = src.to(DEVICE), trg.to(DEVICE)

            output = model(src, trg, teacher_forcing_ratio=0.0)

            output = output[:, 1:].reshape(-1, VOCAB_SIZE)
            trg = trg[:, 1:].reshape(-1)

            loss = criterion(output, trg)
            total_loss += loss.item()

    return total_loss / len(loader)

NUM_EPOCHS = 5

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(train_loader)
    val_loss = evaluate_epoch(val_loader)

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

In [ ]:
def summarize(text):
    model.eval()
    src = torch.tensor([encode(clean_text(text), MAX_ART_LEN)]).to(DEVICE)

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src)
        token = torch.tensor([1]).to(DEVICE)
        summary = []

        for _ in range(MAX_SUM_LEN):
            output, hidden, cell = model.decoder(token, hidden, cell, encoder_outputs)
            top = output.argmax(1)
            if top.item() == 2:
                break
            summary.append(idx2word.get(top.item(), ""))
            token = top

    return " ".join(summary)

#Evaluation

In [ ]:
def evaluate_rouge(articles, references, max_samples=100):
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )

    rouge1, rouge2, rougeL = 0, 0, 0
    count = 0

    for article, ref in zip(articles[:max_samples], references[:max_samples]):
        pred = summarize(article)

        scores = scorer.score(ref, pred)

        rouge1 += scores['rouge1'].fmeasure
        rouge2 += scores['rouge2'].fmeasure
        rougeL += scores['rougeL'].fmeasure
        count += 1

    return {
        "ROUGE-1": rouge1 / count,
        "ROUGE-2": rouge2 / count,
        "ROUGE-L": rougeL / count
    }

rouge_scores = evaluate_rouge(
    test_articles,
    test_summaries,
    max_samples=100  # increase if you have time
)

for k, v in rouge_scores.items():
    print(f"{k}: {v:.4f}")

In [ ]:
def show_examples(n=3):
    for i in range(n):
        print("="*80)
        print("ARTICLE:\n", test_articles[i][:1000], "\n")
        print("REFERENCE SUMMARY:\n", test_summaries[i], "\n")
        print("MODEL SUMMARY:\n", summarize(test_articles[i]), "\n")

show_examples(3)
